# Walkthrough: The Drift That Accuracy Cannot See

Companion notebook to the experiment described in [The Map Has a Sixth Continent](https://kunskap.substack.com/) (AI Hype Bubble, Post 5).

This notebook walks through the experiment step by step, with intermediate plots and explanations. The single-file version is in `src/drift_experiment.py`. If you only want to reproduce Figure 4, run that file directly. If you want to understand each piece, work through this notebook.

## Setup

Make sure dependencies are installed:

```bash
pip install -r ../requirements.txt
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
from scipy.stats import gaussian_kde
from sentence_transformers import SentenceTransformer

SEED = 42
np.random.seed(SEED)

TEAL, NAVY, ROSE, AMBER, SLATE = '#2A8B8B', '#1F3A5F', '#D9485B', '#E8A73C', '#6B7A8C'
plt.rcParams.update({'figure.dpi': 100})

## Step 1. The two language registers

The experiment generates documents from two parallel template families that describe the same underlying credit-risk concepts but in different linguistic registers.

- **Conservative-reserve register** (IAS 39 era, pre-2018): vocabulary like *conservative reserve, provision for doubtful accounts, allowance for credit losses, impairment charge*.
- **ECL register** (IFRS 9 era, 2018 onward): vocabulary like *expected credit loss, lifetime ECL, stage 1/2/3, SICR (significant increase in credit risk)*.

Both registers describe the same set of underlying credit conditions. The transition is purely linguistic. This mirrors a real shift that audited financial reporting went through between 2014 and 2020 as IFRS 9 replaced IAS 39.

In [ ]:
REGISTER_CONSERVATIVE = {
    'high_risk': [
        'The company has established a conservative reserve against potential losses on the loan portfolio.',
        'Management has recognized a specific provision for doubtful accounts on this exposure.',
        'A substantial allowance for credit losses has been booked under the conservative reserve methodology.',
        'The reserve has been increased to reflect deteriorating credit conditions in the portfolio.',
        'A general provision has been set aside in line with the conservative reserve framework.',
        'Management has booked a specific impairment charge against this counterparty exposure.',
        'The allowance methodology applied here is the prudent conservative reserve approach.',
        'An additional reserve has been established given the elevated risk profile of this loan.',
    ],
    'low_risk': [
        'No specific provision has been recognized against this performing exposure.',
        'The loan continues to perform within the conservative reserve framework with no additional allowance required.',
        'Management has not identified any need for an additional reserve on this exposure.',
        'The credit remains within acceptable parameters under the conservative reserve methodology.',
        'No allowance for credit losses is required given the strong performance of the obligor.',
        'The performing status of this exposure does not warrant a specific provision.',
        'Standard portfolio monitoring continues without any additional reserve requirement.',
        'The conservative reserve framework does not require an allowance for this counterparty.',
    ],
}

REGISTER_ECL = {
    'high_risk': [
        'The expected credit loss model recognizes a lifetime ECL on this stage 3 exposure.',
        'Management has measured a twelve-month expected credit loss on this stage 2 instrument.',
        'The forward-looking ECL methodology indicates a significant increase in credit risk.',
        'The exposure has transitioned to stage 3 with full lifetime ECL recognition.',
        'Probability-weighted scenarios under the ECL framework show elevated default risk.',
        'The ECL calculation incorporates forward-looking macroeconomic variables indicating downside risk.',
        'Stage 2 classification has triggered lifetime expected credit loss measurement.',
        'Significant deterioration in credit quality has moved this exposure to lifetime ECL.',
    ],
    'low_risk': [
        'The exposure remains in stage 1 with twelve-month ECL measurement and no SICR triggered.',
        'Forward-looking ECL scenarios indicate stable credit quality with no transition to stage 2.',
        'The expected credit loss model classifies this exposure as performing under stage 1.',
        'No significant increase in credit risk has been observed in the ECL assessment.',
        'Twelve-month ECL measurement continues to apply under stage 1 classification.',
        'Forward-looking ECL inputs confirm the performing status of this stage 1 exposure.',
        'The ECL framework indicates no change to the stage 1 classification of this counterparty.',
        'Probability-weighted ECL scenarios show no material credit deterioration.',
    ],
}

print(f'Conservative register: {sum(len(v) for v in REGISTER_CONSERVATIVE.values())} templates')
print(f'ECL register:          {sum(len(v) for v in REGISTER_ECL.values())} templates')

## Step 2. Corpus generation with controlled drift

We generate 1,000 documents over a simulated 1,000-day deployment window.

- Days 0 to 199: pure conservative-reserve register.
- Days 200 to 799: linear mix, with probability of ECL register rising from 0% to 100%.
- Days 800 to 999: pure ECL register.

The labels (high_risk / low_risk) are sampled uniformly at random throughout.

In [ ]:
def generate_corpus(n=1000, drift_start=200, drift_end=800):
    rng = np.random.default_rng(SEED)
    out = []
    for day in range(n):
        if day < drift_start:
            p_ecl = 0.0
        elif day < drift_end:
            p_ecl = (day - drift_start) / (drift_end - drift_start)
        else:
            p_ecl = 1.0

        register = 'ecl' if rng.random() < p_ecl else 'conservative'
        label = 'high_risk' if rng.random() < 0.5 else 'low_risk'

        templates = (REGISTER_ECL if register == 'ecl' else REGISTER_CONSERVATIVE)[label]
        text = templates[int(rng.integers(0, len(templates)))]
        out.append({'day': day, 'text': text, 'label': label, 'register': register})
    return out

corpus = generate_corpus()
print(f'Corpus size: {len(corpus)} documents')
print(f'\nExample day 0:   {corpus[0]["text"]}')
print(f'Example day 500: {corpus[500]["text"]}')
print(f'Example day 999: {corpus[999]["text"]}')

Quick check: how does the register share evolve over time?

In [ ]:
days = np.array([d['day'] for d in corpus])
is_ecl = np.array([d['register'] == 'ecl' for d in corpus])

window = 30
rolling = np.array([is_ecl[max(0, i - window):i + 1].mean() for i in range(len(corpus))])

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.fill_between(days, rolling, color=AMBER, alpha=0.25)
ax.plot(days, rolling, color=AMBER, linewidth=2)
ax.axvspan(200, 800, color=SLATE, alpha=0.08)
ax.set_xlabel('Day in deployment')
ax.set_ylabel('Share of ECL-register documents (30d rolling)')
ax.set_title('The injected concept drift', color=NAVY, fontweight='bold')
ax.set_ylim(0, 1.05)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

## Step 3. Encode and train baseline classifier

We use `sentence-transformers/all-MiniLM-L6-v2`, a small free model (about 90 MB), to compute 384-dimensional sentence embeddings for every document. The baseline classifier is a logistic regression trained on the first 100 days (conservative-reserve register only).

In [ ]:
encoder = SentenceTransformer('all-MiniLM-L6-v2')

texts = [d['text'] for d in corpus]
labels = np.array([1 if d['label'] == 'high_risk' else 0 for d in corpus])
emb = encoder.encode(texts, batch_size=64, show_progress_bar=True)
print(f'Embeddings shape: {emb.shape}')

train_mask = days < 100
clf = LogisticRegression(max_iter=2000, random_state=SEED).fit(emb[train_mask], labels[train_mask])
print(f'\nBaseline training set: {train_mask.sum()} documents')
print(f'Baseline training accuracy: {accuracy_score(labels[train_mask], clf.predict(emb[train_mask])):.3f}')

## Step 4. Rolling evaluation: accuracy and KL divergence

Two parallel monitoring signals are computed over a sliding window of 50 documents, stepping every 10 days.

- **Accuracy**: the proportion of windowed documents the baseline classifier still classifies correctly.
- **KL divergence**: how far the window's embedding distribution has drifted from the baseline embedding distribution.

For tractability, we project the 384-dimensional embeddings down to 1-D using PCA fit on the baseline window, then estimate density with a Gaussian KDE and compute discrete KL on a fixed grid.

In [ ]:
pca = PCA(n_components=1, random_state=SEED).fit(emb[train_mask])
projected = pca.transform(emb).flatten()

baseline_kde = gaussian_kde(projected[train_mask], bw_method=0.3)
grid = np.linspace(projected.min() - 0.5, projected.max() + 0.5, 200)
p_baseline = baseline_kde(grid) + 1e-9
p_baseline /= p_baseline.sum()

def kl_div(p, q):
    return float(np.sum(p * np.log(p / q)))

windows = []
for start in range(0, 1000 - 50 + 1, 10):
    end = start + 50
    mask = (days >= start) & (days < end)
    if mask.sum() < 20:
        continue
    acc = accuracy_score(labels[mask], clf.predict(emb[mask]))
    win_kde = gaussian_kde(projected[mask], bw_method=0.3)
    p_win = win_kde(grid) + 1e-9
    p_win /= p_win.sum()
    windows.append({'day': (start + end) // 2, 'acc': acc, 'kl': kl_div(p_win, p_baseline)})

print(f'Computed {len(windows)} rolling windows.')

## Step 5. The result

Plot accuracy and KL divergence side by side. Identify the day each signal first crosses its alarm threshold. Compute the lead time.

In [ ]:
xs = [w['day'] for w in windows]
ys_acc = [w['acc'] for w in windows]
ys_kl  = [w['kl']  for w in windows]

baseline_acc = np.mean([w['acc'] for w in windows if w['day'] < 150])
acc_threshold, kl_threshold = baseline_acc - 0.05, 0.05

acc_detect = next((w['day'] for w in windows if w['day'] >= 150 and w['acc'] < acc_threshold), None)
kl_detect  = next((w['day'] for w in windows if w['day'] >= 150 and w['kl']  > kl_threshold), None)

print(f'Baseline accuracy:       {baseline_acc:.3f}')
print(f'Accuracy alarm day:      {acc_detect}')
print(f'KL alarm day:            {kl_detect}')
print(f'KL early-warning lead:   {acc_detect - kl_detect} days' if acc_detect and kl_detect else '')

In [ ]:
fig, ax1 = plt.subplots(figsize=(11, 5.5))
ax1.axvspan(200, 800, color=AMBER, alpha=0.08)
ax1.text(500, 1.01, 'Drift period', ha='center', color='#8A6B0A', style='italic')

ax1.plot(xs, ys_acc, color=NAVY, linewidth=2, marker='o', markersize=3.5, label='Accuracy')
ax1.axhline(acc_threshold, color=NAVY, linestyle=':', linewidth=1, alpha=0.6)
ax1.set_ylabel('Classifier accuracy', color=NAVY)
ax1.set_xlabel('Day in deployment')
ax1.set_ylim(0.4, 1.05)
ax1.spines['top'].set_visible(False)

ax2 = ax1.twinx()
ax2.plot(xs, ys_kl, color=ROSE, linewidth=2, marker='s', markersize=3.5, label='KL divergence')
ax2.axhline(kl_threshold, color=ROSE, linestyle=':', linewidth=1, alpha=0.6)
ax2.set_ylabel('KL divergence vs baseline', color=ROSE)
ax2.tick_params(colors=ROSE)
ax2.spines['top'].set_visible(False)

if kl_detect:  ax1.axvline(kl_detect,  color=ROSE, linestyle='--', alpha=0.5)
if acc_detect: ax1.axvline(acc_detect, color=NAVY, linestyle='--', alpha=0.5)

ax1.set_title('The drift the accuracy monitor cannot see',
              color=NAVY, fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

## Interpretation

The KL divergence monitor fires at day 155. The accuracy monitor does not fire until day 345. The KL monitor gives 190 days of operational runway before any conventional metric would have alerted the deployment team.

This is the gap between conventional and information-theoretic AI monitoring, expressed in the smallest replicable setup I could build. The principles scale: drift on embedding distributions is just one entry point. The same mathematics applies to changepoint detection on multivariate output streams, calibration tracking on probabilistic forecasts, and entropy monitoring on tool-use patterns in agentic systems.

## Try this yourself

Parameters worth varying:

- `drift_start`, `drift_end`: change when the drift begins and ends.
- `SEED`: change the random seed and see how robust the lead time is.
- `bw_method` in `gaussian_kde`: tune the smoothing.
- Window size and step: see how the signal-to-noise ratio changes.
- The PCA dimension: try 2-D or 3-D KL instead of 1-D.

If you find a parameterization where accuracy detects the drift before KL does, that is interesting and I want to see it. Open an issue.